# UNETR Training — PlanetScope RGB Tiles for Glacial Lake Segmentation


This notebook trains **MONAI UNETR** for binary glacial lake segmentation using the same data-loading, loss, metrics, logging, and checkpoint style as your UNet++ notebook.

Notes:
- MONAI's `UNETR` can run in **2D** by setting `spatial_dims=2`.
- UNETR does **not** use SMP-style encoder backbones or ImageNet encoder weights; it is trained from scratch here.
- Your 512×512 tiles are suitable because UNETR uses patch-based ViT features and works best when the image size is divisible by 16.


## Dependencies

In [ ]:
# %pip install -q torch torchvision monai rasterio albumentations pandas numpy


## Configuration

In [1]:

from pathlib import Path
import random

IMAGES_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\subset_750\images")  # change to your imagery folder
LABELS_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\subset_750\labels")  # change to your label folder (same filenames)

# If your files are RGB GeoTIFFs, use (3,2,1) only if band 1=Blue, 2=Green, 3=Red.
# If your files are already saved as RGB in order (R,G,B), change this to (1,2,3).
RGB_BANDS = (3, 2, 1)

OUTPUT_DIR_LOGS = Path(r"D:\Thesis\glacial_lake_detection_thesis\experiments\logs\UNETR")
OUTPUT_DIR_MODELS = Path(r"D:\Thesis\glacial_lake_detection_thesis\models\saved_models\UNETR")
(OUTPUT_DIR_LOGS / "rgb_dataset_10_epochs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR_MODELS / "rgb_dataset_10_epochs").mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-4

# UNETR is not backbone-based like SMP. Try one or more configs.
# For 512x512 images, this default config is safer than the full 768-hidden UNETR on limited GPU memory.
UNETR_CONFIGS = [
    {
        "name": "unetr_fs16_h384_heads6",
        "img_size": (512, 512),
        "feature_size": 16,
        "hidden_size": 384,
        "mlp_dim": 1536,
        "num_heads": 6,
        "dropout_rate": 0.0,
        "norm_name": "batch",       # batch is usually better than instance for 2D RGB batches
        "conv_block": True,
        "res_block": True,
    },
    # Uncomment if you have enough VRAM and want the original larger UNETR-like ViT size.
    # {
    #     "name": "unetr_fs16_h768_heads12",
    #     "img_size": (512, 512),
    #     "feature_size": 16,
    #     "hidden_size": 768,
    #     "mlp_dim": 3072,
    #     "num_heads": 12,
    #     "dropout_rate": 0.0,
    #     "norm_name": "batch",
    #     "conv_block": True,
    #     "res_block": True,
    # },
]

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [2]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 750 pairs.
Train: 600, Val: 150


## Dataset & Dataloaders

In [3]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W)
    return arr

def select_bands(x: np.ndarray, bands=(3,2,1)):
    idx = [b-1 for b in bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, bands=(3,2,1), aug=None, normalize=True):
        self.pairs = pairs
        self.bands = bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img_all = read_geotiff(img_path)
        img = select_bands(img_all, self.bands)  # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        img_hwc = np.transpose(img, (1,2,0))    # (H,W,3)
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, bands=RGB_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGB(val_pairs,   bands=RGB_BANDS, aug=None,   normalize=True)

PIN_MEMORY = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

len(train_ds), len(val_ds)


c:\Users\gg\anaconda3\envs\deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(600, 150)

## Loss & Metrics

In [4]:

import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:,0]
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()
        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))
        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model — MONAI UNETR 2D


In [5]:

import torch
import torch.nn as nn
from torch import optim
import pandas as pd
from datetime import datetime
from monai.networks.nets import UNETR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_unetr_rgb(config: dict, in_channels: int = 3, out_channels: int = 1):
    """Create a 2D MONAI UNETR for binary segmentation."""
    model = UNETR(
        spatial_dims=2,
        in_channels=in_channels,
        out_channels=out_channels,
        img_size=config["img_size"],
        feature_size=config.get("feature_size", 16),
        hidden_size=config.get("hidden_size", 384),
        mlp_dim=config.get("mlp_dim", 1536),
        num_heads=config.get("num_heads", 6),
        dropout_rate=config.get("dropout_rate", 0.0),
        norm_name=config.get("norm_name", "batch"),
        conv_block=config.get("conv_block", True),
        res_block=config.get("res_block", True),
        qkv_bias=config.get("qkv_bias", False),
    )
    return model

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    use_amp = bool(use_amp and torch.cuda.is_available())
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0; n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)[:, 0]  # (B,H,W)
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item(); n_batches += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg: agg[k] /= max(1, n_batches)
    torch.cuda.empty_cache()
    return avg_loss, agg

# Optional sanity check: confirms model input/output shapes before training.
config = UNETR_CONFIGS[0]
_test_model = make_unetr_rgb(config).to(device)
_test_x = torch.randn(1, 3, *config["img_size"]).to(device)
with torch.no_grad():
    _test_y = _test_model(_test_x)
print("Sanity check output shape:", tuple(_test_y.shape))
del _test_model, _test_x, _test_y
torch.cuda.empty_cache()


Device: cuda
Sanity check output shape: (1, 1, 512, 512)


## Training

In [6]:

results_csv = OUTPUT_DIR_LOGS / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","model_name","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for config in UNETR_CONFIGS:
    model_name = config["name"]
    print(f"\n=== Training model: {model_name} ===")
    model = make_unetr_rgb(config, in_channels=3, out_channels=1).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    best_val_iou = -1.0
    best_path = OUTPUT_DIR_MODELS / f"{model_name}_best_val_iou.pt"

    for epoch in range(1, EPOCHS+1):
        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)

        print(f"[{model_name}] Epoch {epoch}/{EPOCHS} — "
              f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
              f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}")

        ts = datetime.utcnow().isoformat()
        pd.DataFrame([
            dict(timestamp=ts, model_name=model_name, epoch=epoch, split="train", loss=train_loss, **train_metrics),
            dict(timestamp=ts, model_name=model_name, epoch=epoch, split="val",   loss=val_loss,   **val_metrics),
        ]).to_csv(results_csv, mode="a", index=False, header=False)

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save({
                "model_name": model_name,
                "config": config,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
                "rgb_bands": RGB_BANDS,
            }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR_MODELS}")



=== Training model: unetr_fs16_h384_heads6 ===


C:\Users\gg\AppData\Local\Temp\ipykernel_1828\4222211219.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\gg\AppData\Local\Temp\ipykernel_1828\4222211219.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[unetr_fs16_h384_heads6] Epoch 1/10 — train IoU=0.2060  val IoU=0.3267  train Loss=1.3458  val Loss=1.2144


C:\Users\gg\AppData\Local\Temp\ipykernel_1828\1416413613.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


[unetr_fs16_h384_heads6] Epoch 2/10 — train IoU=0.3611  val IoU=0.4318  train Loss=1.1513  val Loss=1.0511
[unetr_fs16_h384_heads6] Epoch 3/10 — train IoU=0.3991  val IoU=0.4735  train Loss=1.0443  val Loss=0.9757
[unetr_fs16_h384_heads6] Epoch 4/10 — train IoU=0.4242  val IoU=0.5006  train Loss=0.9548  val Loss=0.8077
[unetr_fs16_h384_heads6] Epoch 5/10 — train IoU=0.4505  val IoU=0.4813  train Loss=0.8748  val Loss=0.7393
[unetr_fs16_h384_heads6] Epoch 6/10 — train IoU=0.4628  val IoU=0.5201  train Loss=0.7964  val Loss=0.7129
[unetr_fs16_h384_heads6] Epoch 7/10 — train IoU=0.4781  val IoU=0.5311  train Loss=0.7392  val Loss=0.6368
[unetr_fs16_h384_heads6] Epoch 8/10 — train IoU=0.4871  val IoU=0.5451  train Loss=0.6810  val Loss=0.6012
[unetr_fs16_h384_heads6] Epoch 9/10 — train IoU=0.5030  val IoU=0.5624  train Loss=0.6277  val Loss=0.5536
[unetr_fs16_h384_heads6] Epoch 10/10 — train IoU=0.5177  val IoU=0.5616  train Loss=0.5774  val Loss=0.5192
Training complete. Logs: D:\Thesis\g

## Load Best Checkpoint for Inference


In [ ]:

# Example: reload the best UNETR checkpoint later
# ckpt_path = OUTPUT_DIR / "models" / "unetr_fs16_h384_heads6_best_val_iou.pt"
# ckpt = torch.load(ckpt_path, map_location=device)
# model = make_unetr_rgb(ckpt["config"], in_channels=3, out_channels=1).to(device)
# model.load_state_dict(ckpt["model_state"])
# model.eval()
